# Treinamento do Modelo de Reconhecimento de Gestos

Este notebook lê o arquivo `hand_landmarks_data.csv` e treina um classificador para identificar os gestos com base nas coordenadas dos pontos das mãos.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle
import os
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Carregamento e Exploração dos Dados

In [ ]:
df = pd.read_csv('hand_landmarks_data.csv')

# Verifica se o arquivo tem cabeçalho. Se a primeira coluna não for 'label', 
# recarregamos sem cabeçalho e atribuímos os nomes manualmente.
if 'label' not in df.columns:
    df = pd.read_csv('hand_landmarks_data.csv', header=None)
    # O MediaPipe Hands gera 21 pontos (x, y, z) -> 63 coordenadas + label = 64 colunas
    columns = ['label']
    for i in range(21):
        columns.extend([f'x{i}', f'y{i}', f'z{i}'])
    df.columns = columns

print(f"Total de amostras: {len(df)}")
print("Amostras por gesto:")
print(df['label'].value_counts())

df.head()

## 2. Preparação dos Dados

Separamos as coordenadas (X) das labels (y).

In [ ]:
X = df.drop('label', axis=1)
y = df['label']

# Divisão em treino e teste (80% treino, 20% teste)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Treino: {X_train.shape[0]} amostras")
print(f"Teste: {X_test.shape[0]} amostras")

## 3. Treinamento do Modelo

Utilizaremos o **Random Forest**, que é excelente para classificação de tabelas e lida bem com relações não-lineares.

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

## 4. Avaliação do Modelo

In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Acurácia: {accuracy * 100:.2f}%")
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))

### Matriz de Confusão
Visualizamos onde o modelo está acertando ou confundindo gestos.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=model.classes_, yticklabels=model.classes_)
plt.xlabel('Predição')
plt.ylabel('Real')
plt.title('Matriz de Confusão')
plt.show()

## 5. Salvando o Modelo

Salvamos o modelo treinado em um arquivo `.pkl` dentro da pasta `models` para ser usado no script de visão computacional em tempo real.

In [ ]:
output_dir = 'models'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'gesture_model.pkl')

# Salva o modelo e a lista de classes
model_data = {
    'model': model,
    'labels': model.classes_
}

with open(output_path, 'wb') as f:
    pickle.dump(model_data, f)

print(f"Modelo salvo com sucesso em: {output_path}!")